In [4]:
import torch
# import wandb
from pathlib import Path
import sys
import os
import tiktoken
import time 
import math

sys.path.append("../src")

from dataset import ShakespearDataset
from torch.utils.data import DataLoader, random_split

from utils import accuracy, count_parameters, model_size
from models import GPT, from_pretrained
from torch.nn import functional as F
import settings as s

In [6]:
model = from_pretrained("cpu", True)
model

loading weights from pretrained gpt2


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): FeedForward(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [9]:
# prefix tokens
enc = tiktoken.get_encoding('gpt2')

model.eval()
num_return_sequences = 5
max_length = 100
tokens = enc.encode("Hello, I'm a language model,")
tokens = torch.tensor(tokens, dtype=torch.long) # (8,)
tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1) # (5, 8)
x = tokens

# generate! right now x is (B, T) where B = 5, T = 8
# set the seed to 42
torch.manual_seed(42)
torch.cuda.manual_seed(42)
while x.shape[1] < max_length:
    # forward the model to get the logits
    with torch.no_grad():
        logits = model(x) # (B, T, vocab_size)
        # take the logits at the last position
        logits = logits[:, -1, :] # (B, vocab_size)
        # get the probabilities
        probs = F.softmax(logits, dim=-1)
        # do top-k sampling of 50 (huggingface pipeline default)
        # topk_probs here becomes (5, 50), topk_indices is (5, 50)
        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
        # select a token from the top-k probabilities
        # note: multinomial does not demand the input to sum to 1
        ix = torch.multinomial(topk_probs, 1) # (B, 1)
        # gather the corresponding indices
        xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
        # append to the sequence
        x = torch.cat((x, xcol), dim=1)

# print the generated text
for i in range(num_return_sequences):
    tokens = x[i, :max_length].tolist()
    decoded = enc.decode(tokens)
    print(">", decoded)

> Hello, I'm a language model, not a programming platform! I just make a language and learn things. I make things that look something like I'm developing an app, not like I'm being forced into this.

"The thing is the programming model is so abstract, and the code and the syntax is so simple – so that when I'm able to write something, that's what it is, and the programmers are making an interesting language. And if I can code that well,
> Hello, I'm a language model, a kind of a "first class citizen" of the world and a person that comes from a much more special place than that. But we're in our late 20s, and I hope that because of the fact that we're very aware of our language being limited there are ways to make that possible.

Q: Is it okay if you get paid for this?

MR. NOMOS, HOST:

This is FRESH
> Hello, I'm a language model, and I'm starting to talk about the notion of the monads. I'm a good beginner and I know the concept in advance. But when I start to speak out on the history 

In [2]:
data_path = Path("../data")
logs_path = Path("../logs")
logs_path.mkdir(exist_ok=True)

In [3]:
# wandb.init(
#     project=s.project_name,
#     config={
#         "model": s.model,
#         "dataset": s.dataset,
#         "max_epochs": s.max_epochs,
#         "optimizer": s.optimizer,
#         "test_run": s.test_run,
#     },
#     dir=logs_path,
#     mode="offline" if s.wandb_offline else "online"
# )

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
# cpu_count = os.cpu_count()
cpu_count = 7

dataset = ShakespearDataset(data_path/"shakespear.txt")

train_dataset, val_dataset = random_split(
    dataset, [s.dataset["train_split"], s.dataset["val_split"]]
)

train_dataloader = DataLoader(
    train_dataset, batch_size=s.dataset["batch_size"], shuffle=True, num_workers=cpu_count)
val_dataloader = DataLoader(
    val_dataset, batch_size=s.dataset["batch_size"],  num_workers=cpu_count)

In [5]:
model = GPT(device).to(device)
model = torch.compile(model)

model_size(model)
count_parameters(model)

Model size: 523.03 MB
Trainable parameters: 124.53M
Non-trainable parameters: 0


In [6]:
max_lr = 6e-4
min_lr = max_lr * 0.1
warmup_steps = 10
max_steps = 50
def get_lr(it):
    # 1) linear warmup for warmup_iters steps
    if it < warmup_steps:
        return max_lr * (it+1) / warmup_steps

    # 2) if it > lr_decay_iters, return min learning rate
    if it > max_steps:
        return min_lr

    # 3) in between, use cosine decay down to min learning rate
    decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff starts at 1 and goes to 0
    
    return min_lr + coeff * (max_lr - min_lr)

In [7]:
optimizer = model.configure_optimizers(weight_decay=0.1, learning_rate=6e-4, device=device)

assert s.dataset["total_batch_size"] % (s.dataset["batch_size"] * s.dataset["context_size"]) == 0, "make sure total_batch_size is divisible by B * T"
grad_accum_steps = s.dataset["total_batch_size"] // (s.dataset["batch_size"] * s.dataset["context_size"])
print(f"=> calculated gradient accumulation steps: {grad_accum_steps}")

train_dataloader_iter = iter(train_dataloader)

num decayed parameter tensors: 50, with 124,354,560 parameters
num non-decayed parameter tensors: 99, with 171,648 parameters
using fused AdamW: True
=> calculated gradient accumulation steps: 16


In [8]:
sms = torch.cuda.get_device_properties(0).multi_processor_count
sms

108

In [9]:
max_steps = 10

for epoch in range(s.max_epochs):
    print(f"Epoch: {epoch} \n")

    for step in range(max_steps):
        t0 = time.time()
        optimizer.zero_grad()
        step_loss = 0.0
        for micro_step in range(grad_accum_steps):
            x, y = next(train_dataloader_iter)
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device, dtype=torch.bfloat16):
                logits = model(x)
                loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))

            # we have to scale the loss to account for gradient accumulation,
            # because the gradients just add on each successive backward().
            # addition of gradients corresponds to a SUM in the objective, but
            # instead of a SUM we want MEAN. Scale the loss here so it comes out right
            loss = loss / grad_accum_steps
            step_loss += loss.detach()
            loss.backward()

        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        # determine and set the learning rate for this iteration
        lr = get_lr(step)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        optimizer.step()
        torch.cuda.synchronize() # wait for the GPU to finish work
        t1 = time.time()
        dt = t1 - t0 # time difference in seconds
        tokens_processed = s.dataset["batch_size"] * s.dataset["context_size"] * grad_accum_steps
        tokens_per_sec = tokens_processed / dt
        
        print(f"step {step:4d} | loss: {step_loss:.4f} | norm: {norm:.4f} | lr:{lr:.4f} | dt: {dt*1000:.2f}ms | tok/sec: {tokens_per_sec:.2f}")

Epoch: 0 



step    0 | loss: 11.0341 | norm: 2.9400 | lr:0.0001 | dt: 28557.75ms | tok/sec: 18358.87
step    1 | loss: 10.2923 | norm: 3.1060 | lr:0.0001 | dt: 3453.20ms | tok/sec: 151826.70
step    2 | loss: 9.4762 | norm: 2.3936 | lr:0.0002 | dt: 3457.29ms | tok/sec: 151647.24
step    3 | loss: 9.0293 | norm: 1.8819 | lr:0.0002 | dt: 3456.76ms | tok/sec: 151670.49
step    4 | loss: 8.8204 | norm: 3.2051 | lr:0.0003 | dt: 3453.35ms | tok/sec: 151819.98
step    5 | loss: 8.4893 | norm: 2.0830 | lr:0.0004 | dt: 3457.78ms | tok/sec: 151625.67
step    6 | loss: 8.2648 | norm: 3.1535 | lr:0.0004 | dt: 3460.71ms | tok/sec: 151497.17
step    7 | loss: 7.9294 | norm: 1.9312 | lr:0.0005 | dt: 3456.71ms | tok/sec: 151672.45
step    8 | loss: 7.6013 | norm: 1.9114 | lr:0.0005 | dt: 3454.07ms | tok/sec: 151788.34
step    9 | loss: 7.2984 | norm: 2.0652 | lr:0.0006 | dt: 3453.00ms | tok/sec: 151835.39


In [ ]:
# min sms=80
# [sms: 108] [A100] step    9 | loss: 7.2984 | norm: 2.0652 | lr:0.0006 | dt: 3453.00ms | tok/sec: 151835.39 | 105
# [sms: 84] [A6000] step    9 | loss: 7.2137 | norm: 1.3000 | lr:0.0006 | dt: 6834.21ms | tok/sec: 76715.26 | 67
# [sms: 64] [A5000] step    9 | loss: 7.2747 | norm: 1.7772 | lr:0.0006 | dt: 8048.46ms | tok/sec: 65141.38 | 36


In [ ]:
# steps = 20_000
# Af_cost = 8 * steps / (3600) * 36
# print(Af_cost)

# As_cost = 7 * steps / (3600) * 67
# print(As_cost)

# Ah_cost = 3.5 * steps / (3600) * 105
# print(As_cost)

1600.0
2605.555555555555
2605.555555555555


In [11]:
# text = """First Citizen:
# Very well;"""

# tokens = torch.tensor(dataset.encoder.encode(text))
# s.dataset["vocab_size"] = dataset.encoder.n_vocab

# x = tokens.unsqueeze(dim=0).repeat(2, 1).to(model.device)
# model.eval()
# while x.shape[1] < 30: 
#     with torch.no_grad():
#         logits = model(x) 
#         logits = logits[:, -1, :] 
#         probs = F.softmax(logits, dim=-1)
#         topk_probs, topk_indices = torch.topk(probs, 50, dim=-1) 
#         ix = torch.multinomial(topk_probs, 1) 
#         xcol = torch.gather(topk_indices, -1, ix) 
#         x = torch.cat((x, xcol), dim=1)

# for row in x:
#     print(">", dataset.encoder.decode(row.tolist()))